# Drive -> repo, from Colab

**Route 4: nothing installed locally.** This notebook copies the Google Drive files that are still
missing from `lach-matt/Claude-Method-Works` into the repo and pushes them, using only a browser.
It is Route 4 in [`docs/DRIVE-SYNC.md`](../docs/DRIVE-SYNC.md) §2, and the numbering here matches.

It exists because the other routes each need something you may not want to set up:

| Route | Needs | Ceiling |
| --- | --- | --- |
| 1 - Claude Drive connector | nothing | **~6 MiB per file** - 34 files (at time of writing) cannot come through it |
| 2 - `tools/drive_sync.py` locally | Google Cloud project, OAuth desktop client, Python locally | none |
| 3 - `drive_sync.py` on a GitHub Actions schedule | Route 2's setup once, then a repository secret | none |
| **4 - this notebook** | a GitHub token, and one click to authorise Drive | none |

Colab mounts *your own* Drive with `google.colab.drive.mount`, so the files are just files on a
filesystem. No Drive API credentials, no OAuth client, no payload ceiling.

## What you need

1. **A GitHub personal access token with `repo` scope.**
   Fine-grained tokens work too: give it *Contents: read and write* on `lach-matt/Claude-Method-Works`.
   Create one at <https://github.com/settings/tokens>. Prefer a **short expiry** - you are pasting it
   into a cloud VM. Cell 2 reads it with `getpass`, so it is never typed into a visible field, never
   printed, and never written into `.git/config`.
2. **The Drive account that owns the two Method folders**, to authorise the mount in cell 1.

## Warnings, plainly

* **This commits large binary files.** The outstanding set includes multi-megabyte `.md` builds, PDFs,
  zips and JSON exports. They go into git history permanently - a later `git rm` does not shrink the
  repo, only a history rewrite does. See `docs/REPO-SIZE.md`.
* **The two 370 MiB `conversations.json` exports are deliberately not copied.** They exceed GitHub's
  hard 100 MB per-file limit and can never be ordinary git objects. Cell 3 skips any row whose
  recorded `drive_size_bytes` is at or over 100 MB. The appendix at the bottom md5s **whatever copies
  the Drive mount exposes** - which is normally only one, because a filesystem cannot show two entries
  with the same name in one folder, so it usually *cannot* settle whether the two Drive files are the
  same bytes. Answering that needs the Drive API (`tools/drive_sync.py`) or the Drive web UI. The
  appendix can also hand one copy to `tools/shard_conversations.py` if you decide the conversations
  belong in the repo.
* **`INCLUDE_HELD_BACK` in cell 3 currently matches nothing.** It filters on the phrase *held back* in
  `PENDING.tsv`'s `reason` column, and no row says that today: the Claude account records
  (`Claude Metadata/users.json`, `login_history.json`, which carry login IPs and user identity) are not
  listed in `PENDING.tsv` at all. Fetch those deliberately with
  `python3 tools/drive_sync.py --only "Claude Metadata"` instead.
* **The mirror is one-way.** Nothing here writes to Drive. Files are only ever read from the mount.
* **Run the cells in order, once each.** Every cell checks its own preconditions and stops with a
  readable message rather than guessing, so a cell run out of order will tell you what to run first.


## 1. Mount Drive and find the two source folders

Running this pops up an authorisation dialog. Pick the account that owns the Method folders and allow
access; Colab mounts it read-write at `/content/drive`, and your files live under
`/content/drive/MyDrive/`.

**The trailing-space trap.** The real Drive title of the first folder is `"The Method Materials "` -
with a **trailing space**. The repo strips it (`drive/The Method Materials/`), so the two names do not
match literally. Any shell path you type by hand must therefore be quoted, and may need that trailing
space:

```bash
ls -la "/content/drive/MyDrive/The Method Materials "     # note the space before the quote
```

The cell below does not make you guess: it looks for an exact match first, then for a folder whose
name matches once you ignore surrounding whitespace and dots, and prints the resolved path with
`repr()` so a trailing space is visible as `'The Method Materials '`.


In [ ]:
# Cell 1 - mount Drive, resolve the two source folders, count what is in them.
import os
import unicodedata

DRIVE_MOUNT = "/content/drive"
DRIVE_ROOT = os.path.join(DRIVE_MOUNT, "MyDrive")

# Drive titles as they appear in the repo (i.e. with the trailing space stripped).
# Matching below is whitespace-insensitive, so "The Method Materials " resolves too.
SOURCE_FOLDER_NAMES = (
    "The Method Materials",
    "The Method Prints & Proofs",
)


def norm_name(name):
    """Fold a Drive title to the form used for matching: NFC, no edge space/dots, casefolded."""
    return unicodedata.normalize("NFC", name).strip().strip(".").casefold()


def resolve_child(parent, name):
    """Return the real path of `name` inside `parent`, tolerating edge whitespace/dots/case.

    Returns None when nothing matches. Raises when several entries match, because guessing
    between two folders that differ only by a trailing space would silently mirror the wrong one.
    """
    exact = os.path.join(parent, name)
    if os.path.exists(exact):
        return exact
    try:
        entries = os.listdir(parent)
    except OSError as exc:
        raise RuntimeError("cannot list {!r}: {}".format(parent, exc))
    wanted = norm_name(name)
    hits = sorted(entry for entry in entries if norm_name(entry) == wanted)
    if len(hits) == 1:
        return os.path.join(parent, hits[0])
    if len(hits) > 1:
        raise RuntimeError(
            "{!r} is ambiguous inside {!r} - {} entries match after normalisation: {}. "
            "Rename or remove one in Drive, or hard-code the path you want.".format(
                name, parent, len(hits), hits
            )
        )
    return None


def count_tree(path):
    """(files, directories, total bytes) under `path`. Unreadable files are reported, not skipped."""
    files = dirs = 0
    total = 0
    problems = []
    for root, dirnames, filenames in os.walk(path):
        dirs += len(dirnames)
        files += len(filenames)
        for filename in filenames:
            try:
                total += os.path.getsize(os.path.join(root, filename))
            except OSError as exc:
                problems.append("{}: {}".format(os.path.join(root, filename), exc))
    return files, dirs, total, problems


if not os.path.ismount(DRIVE_MOUNT) and not os.path.isdir(DRIVE_ROOT):
    try:
        from google.colab import drive as colab_drive  # only exists inside Colab
    except ImportError as exc:
        raise RuntimeError(
            "google.colab is not importable, so this is not a Colab runtime. This notebook only "
            "works in Colab (https://colab.research.google.com). To sync from your own machine "
            "instead, use tools/drive_sync.py - see docs/DRIVE-SYNC.md, Route 2."
        ) from exc

    colab_drive.mount(DRIVE_MOUNT)
else:
    print("Drive already mounted at {}".format(DRIVE_MOUNT))

if not os.path.isdir(DRIVE_ROOT):
    raise RuntimeError(
        "{} does not exist after mounting. Either the mount was cancelled, or this Drive is "
        "organised differently (shared drives live under {}/Shareddrives). "
        "Run  !ls -la /content/drive  to see what is actually there.".format(
            DRIVE_ROOT, DRIVE_MOUNT
        )
    )

SOURCE_ROOTS = {}
missing = []
for wanted in SOURCE_FOLDER_NAMES:
    found = resolve_child(DRIVE_ROOT, wanted)
    if found is None:
        missing.append(wanted)
    else:
        SOURCE_ROOTS[wanted] = found

if missing:
    top_level = sorted(entry for entry in os.listdir(DRIVE_ROOT))
    raise RuntimeError(
        "Could not find {} under {!r}.\n"
        "These folders ARE there (repr() shown so trailing spaces are visible):\n  {}\n"
        "Fix SOURCE_FOLDER_NAMES above to match, or check you authorised the right "
        "Google account. Shared drives are not under MyDrive - look in {}/Shareddrives.".format(
            ", ".join(repr(name) for name in missing),
            DRIVE_ROOT,
            "\n  ".join(repr(entry) for entry in top_level) or "(nothing)",
            DRIVE_MOUNT,
        )
    )

print("Resolved source folders (repr() so a trailing space is visible):")
for wanted, path in SOURCE_ROOTS.items():
    print("  {!r}\n    -> {!r}".format(wanted, path))

print("\nCounting files (walking a Drive mount is slow the first time; give it a minute)...")
for wanted, path in SOURCE_ROOTS.items():
    files, dirs, total, problems = count_tree(path)
    print(
        "  {!r}: {} files in {} subfolders, {:.1f} MiB".format(
            wanted, files, dirs, total / 1048576.0
        )
    )
    for problem in problems:
        print("    UNREADABLE: {}".format(problem))

print("\nOK - Drive is mounted and both folders are visible.")


## 2. Clone the repository

The token is read with `getpass` (hidden input) and handed to git through a **credential store file**
at `/content/.git-credentials`, mode `0600`.

That indirection is the point. The obvious alternative -
`git clone https://TOKEN@github.com/...` - leaves the token in three places that outlive the cell:
`.git/config`, the process list, and any error message git prints. Using a credential file keeps the
remote URL clean (`https://github.com/lach-matt/Claude-Method-Works`), and cell 6 deletes the file
after a successful push. Nothing in this notebook ever prints the token: every subprocess's output is
scrubbed before it is shown, in case git echoes a URL back at you.

Re-running this cell is safe. If `/content/repo` already exists it is reused (fetch + checkout)
rather than re-cloned, so work from an earlier attempt is not thrown away.


In [ ]:
# Cell 2 - clone (or refresh) the repo at /content/repo on the working branch.
import getpass
import os
import pathlib
import subprocess
import urllib.parse

REPO_SLUG = "lach-matt/Claude-Method-Works"
REPO_URL = "https://github.com/{}.git".format(REPO_SLUG)
BRANCH = "claude/google-drive-github-sync-3833rb"
REPO_DIR = "/content/repo"
CRED_FILE = "/content/.git-credentials"

# Used as the commit author. Change if you are not the repo owner.
GIT_USER_NAME = "lach-matt"
GIT_USER_EMAIL = "lach.matthew@gmail.com"

_SECRETS = []  # never printed; used to scrub subprocess output


def scrub(text):
    """Replace anything secret with *** before it can reach the notebook output."""
    if not text:
        return ""
    for secret in _SECRETS:
        if secret:
            text = text.replace(secret, "***")
    return text


def git(*args, cwd=None, check=True):
    """Run git, returning the CompletedProcess. Raises with scrubbed output on failure."""
    proc = subprocess.run(
        ("git",) + args, cwd=cwd, text=True, capture_output=True
    )
    if check and proc.returncode != 0:
        raise RuntimeError(
            "git {} failed (exit {}):\n{}\n{}".format(
                scrub(" ".join(args)), proc.returncode, scrub(proc.stdout).strip(), scrub(proc.stderr).strip()
            )
        )
    return proc


token = getpass.getpass(
    "GitHub personal access token (repo scope) - input is hidden, nothing is echoed: "
).strip()
if not token:
    raise RuntimeError("No token entered. Re-run this cell and paste the token.")
if len(token) < 20 and not token.startswith(("ghp_", "github_pat_")):
    raise RuntimeError(
        "That does not look like a GitHub token (under 20 characters). Nothing was stored - "
        "re-run this cell and paste the whole token."
    )
_SECRETS.append(token)

# Credential store file: mode 0600, created before anything is written into it.
fd = os.open(CRED_FILE, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
with os.fdopen(fd, "w") as handle:
    handle.write(
        "https://{}:{}@github.com\n".format(
            urllib.parse.quote(GIT_USER_NAME, safe=""), urllib.parse.quote(token, safe="")
        )
    )
del token  # the value survives only inside CRED_FILE and _SECRETS from here on

git("config", "--global", "credential.helper", "store --file={}".format(CRED_FILE))

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    existing = git("-C", REPO_DIR, "config", "--get", "remote.origin.url").stdout.strip()
    if REPO_SLUG.lower() not in existing.lower():
        raise RuntimeError(
            "{} already exists but its origin is {!r}, not {}. Move it aside "
            "(!mv /content/repo /content/repo.old) and re-run.".format(
                REPO_DIR, scrub(existing), REPO_SLUG
            )
        )
    print("Reusing existing clone at {}".format(REPO_DIR))
    git("-C", REPO_DIR, "fetch", "origin", BRANCH)
    current = git("-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD").stdout.strip()
    if current != BRANCH:
        git("-C", REPO_DIR, "checkout", BRANCH)
else:
    print("Cloning {} (branch {})...".format(REPO_SLUG, BRANCH))
    git("clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR)

git("-C", REPO_DIR, "config", "user.name", GIT_USER_NAME)
git("-C", REPO_DIR, "config", "user.email", GIT_USER_EMAIL)

# The remote URL must be token-free, or the token would live on in .git/config.
config_text = pathlib.Path(REPO_DIR, ".git", "config").read_text(encoding="utf-8")
if "@github.com" in config_text:
    raise RuntimeError(
        ".git/config contains a credential in the remote URL. Fix it with:\n"
        "  !git -C {} remote set-url origin {}".format(REPO_DIR, REPO_URL)
    )

head = git("-C", REPO_DIR, "log", "-1", "--pretty=%h %s").stdout.strip()
branch_now = git("-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD").stdout.strip()
pending_path = os.path.join(REPO_DIR, "drive", "PENDING.tsv")
if not os.path.isfile(pending_path):
    raise RuntimeError(
        "{} is missing. Is {} really the branch that carries the mirror?".format(pending_path, BRANCH)
    )
with open(pending_path, encoding="utf-8") as handle:
    pending_rows = sum(1 for _ in handle) - 1

print("\nrepo:    {}".format(REPO_DIR))
print("remote:  {}".format(scrub(git("-C", REPO_DIR, "config", "--get", "remote.origin.url").stdout.strip())))
print("branch:  {}".format(branch_now))
print("head:    {}".format(head))
print("author:  {} <{}>".format(GIT_USER_NAME, GIT_USER_EMAIL))
print("pending: {} rows in drive/PENDING.tsv".format(pending_rows))
print("token:   held in {} (mode 600), not in .git/config".format(CRED_FILE))


## 3. Copy the missing files in

`drive/PENDING.tsv` already lists every outstanding file, so this cell is driven entirely by it -
no Drive API call, no second inventory to keep in step. Its six columns are
`repo_path, drive_id, drive_title, drive_size_bytes, source, reason`.

For each row the cell resolves `drive/<repo_path>` back to a real path under the mount, walking the
folder components with the same whitespace-tolerant matching as cell 1 (Drive subfolders can carry
trailing spaces too), then copies the file to `<repo>/drive/<repo_path>`, creating parent directories.
Copies land on a `.part` file first and are then renamed into place, so an interrupted run never
leaves a truncated file that looks finished.

**Rows it deliberately does not copy**

* `drive_size_bytes` is at or over 100,000,000 -> the two 370 MiB `conversations.json` exports. They
  cannot be committed at all; see the appendix cell at the bottom. **The size is the test**, because
  `reason` is hand-written prose: the `reason` regex is kept only as an extra trigger, so a reworded
  reason cannot disarm the guard and let 370 MiB into git history.
* `reason` says *held back* -> Claude account records (login IPs, account identity). Set
  `INCLUDE_HELD_BACK = True` below only if you have decided you want those in a GitHub repo.
  **No row says this today**, so the flag currently matches nothing; `PENDING.tsv` is hand-maintained
  (no tool writes it), and the Claude Metadata files are fetched with
  `python3 tools/drive_sync.py --only "Claude Metadata"`.

**Two titles are duplicated in Drive** (`conversations.json`, and
`The_Method_1_6_BUILD178_compendia_papers_audits.md`, whose two copies are both 5,619,374 bytes). The
mirror disambiguates them by appending `__<driveFileId>` to the older copy's filename, but a mounted
filesystem exposes no Drive ids, so this cell cannot tell you which copy is which. Where that happens
it matches candidates in sorted order and says so loudly rather than pretending to know.


In [ ]:
# Cell 3 - copy every outstanding file from the mount into the clone.
import csv
import json
import os
import re
import shutil
import unicodedata

REPO_DIR = "/content/repo"
DRIVE_ROOT = "/content/drive/MyDrive"
STATE_PATH = "/content/drive_sync_colab_state.json"
PENDING_PATH = os.path.join(REPO_DIR, "drive", "PENDING.tsv")
DEST_ROOT = os.path.join(REPO_DIR, "drive")

# Copy the two Claude account-record files (login IPs, user identity)? Off by default.
# NOTE: no PENDING.tsv row says "held back" today - the Claude Metadata files are not
# listed in PENDING.tsv at all - so this flag currently matches nothing either way. The
# way to fetch them is:  python3 tools/drive_sync.py --only "Claude Metadata"
INCLUDE_HELD_BACK = False

REQUIRED_COLUMNS = ("repo_path", "drive_id", "drive_title", "drive_size_bytes", "reason")

# The real guard against committing a file GitHub will reject is the byte count, not the
# prose in `reason`: PENDING.tsv is hand-maintained, so a reworded reason must not be able
# to let a 370 MiB blob into git history (where only a rewrite could remove it again).
# OVERSIZE_RE stays as a second, independent trigger.
GITHUB_FILE_LIMIT_BYTES = 100 * 1000 * 1000
OVERSIZE_RE = re.compile(r"100\s*M(?:B|iB)", re.IGNORECASE)
HELD_BACK_RE = re.compile(r"held back", re.IGNORECASE)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    raise RuntimeError("No clone at {}. Run cell 2 first.".format(REPO_DIR))
if not os.path.isdir(DRIVE_ROOT):
    raise RuntimeError("Drive is not mounted at {}. Run cell 1 first.".format(DRIVE_ROOT))


def norm_name(name):
    return unicodedata.normalize("NFC", name).strip().strip(".").casefold()


def resolve_child(parent, name):
    """Real path of `name` under `parent`, tolerating edge whitespace/dots/case. None if absent."""
    exact = os.path.join(parent, name)
    if os.path.exists(exact):
        return exact
    try:
        entries = os.listdir(parent)
    except OSError:
        return None
    wanted = norm_name(name)
    hits = sorted(entry for entry in entries if norm_name(entry) == wanted)
    if len(hits) == 1:
        return os.path.join(parent, hits[0])
    if len(hits) > 1:
        raise RuntimeError(
            "{!r} matches {} entries inside {!r}: {}".format(name, len(hits), parent, hits)
        )
    return None


_dir_cache = {}


def resolve_dir(components):
    """Walk repo_path's directory components down the mount. None if any component is missing."""
    key = tuple(components)
    if key in _dir_cache:
        return _dir_cache[key]
    current = DRIVE_ROOT
    for component in components:
        current = resolve_child(current, component)
        if current is None or not os.path.isdir(current):
            _dir_cache[key] = None
            return None
    _dir_cache[key] = current
    return current


def candidates_for(folder, title):
    """Every file in `folder` whose name matches `title` after normalisation, sorted."""
    try:
        entries = os.listdir(folder)
    except OSError as exc:
        raise RuntimeError("cannot list {!r}: {}".format(folder, exc))
    wanted = norm_name(title)
    return sorted(
        os.path.join(folder, entry)
        for entry in entries
        if norm_name(entry) == wanted and os.path.isfile(os.path.join(folder, entry))
    )


with open(PENDING_PATH, encoding="utf-8", newline="") as handle:
    reader = csv.reader(handle, delimiter="\t", quotechar=None, quoting=csv.QUOTE_NONE)
    header = next(reader)
    absent = [column for column in REQUIRED_COLUMNS if column not in header]
    if absent:
        raise RuntimeError(
            "PENDING.tsv is missing the column(s) {}.\n  header found: {}\n"
            "This cell reads columns by name; it will not guess at a changed layout.".format(
                absent, header
            )
        )
    rows = []
    for number, row in enumerate(reader, start=2):
        if not row:
            continue
        if len(row) != len(header):
            raise RuntimeError(
                "PENDING.tsv line {} has {} fields, header has {}. Refusing to read a file whose "
                "rows do not line up with its header.".format(number, len(row), len(header))
            )
        rows.append(dict(zip(header, row)))

print("{} rows in PENDING.tsv".format(len(rows)))
if INCLUDE_HELD_BACK and not any(HELD_BACK_RE.search(row["reason"]) for row in rows):
    print(
        "NOTE: INCLUDE_HELD_BACK is True but no row's reason says 'held back', so it changes\n"
        "      nothing. The Claude account records are fetched with:\n"
        '        python3 tools/drive_sync.py --only "Claude Metadata"'
    )

# Group by (directory, drive_title) so duplicate titles are handled as a set, not one at a time.
groups = {}
skipped = []
for row in rows:
    reason = row["reason"]
    # Parsed here, before any grouping, so the size guard runs on every single row.
    try:
        row["_expected_size"] = int(row["drive_size_bytes"])
    except ValueError:
        raise RuntimeError(
            "row {!r} has a non-numeric drive_size_bytes {!r}".format(
                row["repo_path"], row["drive_size_bytes"]
            )
        )
    if row["_expected_size"] >= GITHUB_FILE_LIMIT_BYTES:
        skipped.append(
            (
                row["repo_path"],
                "{:,} bytes - at or over GitHub's 100 MB per-file limit; see the appendix cell".format(
                    row["_expected_size"]
                ),
            )
        )
        continue
    if OVERSIZE_RE.search(reason):
        skipped.append((row["repo_path"], "reason names GitHub's 100 MB limit - see the appendix cell"))
        continue
    if HELD_BACK_RE.search(reason) and not INCLUDE_HELD_BACK:
        skipped.append((row["repo_path"], "held back (set INCLUDE_HELD_BACK = True to copy)"))
        continue
    parts = row["repo_path"].split("/")
    groups.setdefault((tuple(parts[:-1]), row["drive_title"]), []).append(row)

copied, present, missing, ambiguous = [], [], [], []

for (components, title), members in sorted(groups.items()):
    members = sorted(members, key=lambda item: item["repo_path"])
    folder = resolve_dir(components)
    if folder is None:
        for row in members:
            missing.append((row["repo_path"], "folder {!r} not found under the mount".format("/".join(components))))
        continue

    found = candidates_for(folder, title)
    if len(members) > 1 or len(found) > 1:
        ambiguous.append(
            "{!r} in {!r}: {} PENDING row(s), {} file(s) on the mount. Drive ids are invisible "
            "to a filesystem, so copies are paired in sorted order - verify sizes/md5 below.".format(
                title, "/".join(components), len(members), len(found)
            )
        )

    for index, row in enumerate(members):
        target = os.path.join(DEST_ROOT, row["repo_path"])
        expected = row["_expected_size"]  # parsed and range-checked above

        if os.path.isfile(target) and os.path.getsize(target) == expected:
            present.append(row["repo_path"])
            continue

        if index >= len(found):
            missing.append(
                (
                    row["repo_path"],
                    "no file named {!r} left in {!r} ({} needed, {} present)".format(
                        title, folder, len(members), len(found)
                    ),
                )
            )
            continue

        source = found[index]
        os.makedirs(os.path.dirname(target), exist_ok=True)
        partial = target + ".part"
        shutil.copyfile(source, partial)  # raises on any read/write error; nothing is swallowed
        os.replace(partial, target)
        copied.append(
            {
                "repo_path": row["repo_path"],
                "drive_id": row["drive_id"],
                "drive_title": title,
                "expected_size": expected,
                "source": source,
            }
        )

for item in copied:
    print("copied         {}".format(item["repo_path"]))
for repo_path in present:
    print("already present {}".format(repo_path))
for repo_path, why in missing:
    print("NOT FOUND      {}  ({})".format(repo_path, why))
for repo_path, why in skipped:
    print("skipped        {}  ({})".format(repo_path, why))
if ambiguous:
    print("\nDuplicate-title warnings:")
    for note in ambiguous:
        print("  {}".format(note))

state = {
    "repo_dir": REPO_DIR,
    "copied": copied,
    "present": present,
    "missing": missing,
    "skipped": skipped,
    "ambiguous": ambiguous,
    "verified": False,
}
with open(STATE_PATH, "w", encoding="utf-8") as handle:
    json.dump(state, handle, indent=1)

print(
    "\n{} copied, {} already present, {} not found, {} skipped. State written to {}".format(
        len(copied), len(present), len(missing), len(skipped), STATE_PATH
    )
)
if missing:
    print("NOT FOUND rows are listed above and will NOT be committed. Nothing is hidden.")


## 4. Verify every copied file against its recorded Drive size

A byte count is a weak checksum but a strong smoke test, and it is the only check available here:
`PENDING.tsv` carries `drive_size_bytes` but no md5.

It catches the failure modes that actually happen on a Drive mount:

* a truncated copy from a dropped connection,
* a Google-native doc, which appears on the mount as a ~200-byte `.gdoc` JSON pointer rather than the
  document (the mirror expects an *export* of those, which a mount cannot produce),
* the wrong copy of a duplicated title, when the two copies differ in size.

**Any mismatch stops the run here.** The commit cell refuses to proceed unless this cell has set
`verified: true` in the state file.


In [ ]:
# Cell 4 - size-verify everything cell 3 copied. Refuses to pass on any mismatch.
import json
import os

STATE_PATH = "/content/drive_sync_colab_state.json"
REPO_DIR = "/content/repo"
DEST_ROOT = os.path.join(REPO_DIR, "drive")

if not os.path.isfile(STATE_PATH):
    raise RuntimeError("No state file at {}. Run cell 3 (the copy cell) first.".format(STATE_PATH))
with open(STATE_PATH, encoding="utf-8") as handle:
    state = json.load(handle)

copied = state.get("copied", [])
if not copied:
    print("Cell 3 copied nothing, so there is nothing to verify.")
    if state.get("missing"):
        print("It did report {} file(s) NOT FOUND - see cell 3's output.".format(len(state["missing"])))

mismatches = []
for item in copied:
    target = os.path.join(DEST_ROOT, item["repo_path"])
    if not os.path.isfile(target):
        mismatches.append((item["repo_path"], item["expected_size"], "file is gone"))
        continue
    actual = os.path.getsize(target)
    if actual != item["expected_size"]:
        mismatches.append((item["repo_path"], item["expected_size"], actual))

for repo_path, expected, actual in mismatches:
    print("SIZE MISMATCH  {}\n    expected {} bytes, got {}".format(repo_path, expected, actual))

state["verified"] = not mismatches
with open(STATE_PATH, "w", encoding="utf-8") as handle:
    json.dump(state, handle, indent=1)

if mismatches:
    raise RuntimeError(
        "{} of {} copied file(s) do not match drive_size_bytes. Nothing will be committed.\n"
        "Likely causes: an interrupted copy (delete the file and re-run cell 3); a Google-native "
        "doc that the mount exposes as a small .gdoc/.gsheet pointer (use tools/drive_sync.py, "
        "which exports those properly); or the wrong copy of a duplicated title.".format(
            len(mismatches), len(copied)
        )
    )

print("OK - all {} copied file(s) match the size recorded in PENDING.tsv.".format(len(copied)))


## 5. Manifest rows for the new files

`drive/MANIFEST.tsv` is the 8-column inventory of everything mirrored
(`repo_path drive_id drive_title mime_type drive_size_bytes drive_modified md5 status`).

**This notebook cannot regenerate it properly, and does not pretend to.** The obvious move -
"just run `tools/drive_sync.py`" - does not work here: that script talks to the Drive API and needs
the OAuth client this whole route exists to avoid. Two of the eight columns are also simply not
knowable from a filesystem mount: `mime_type` is guessed from the extension, and Drive's
`modifiedTime` is not the mount's mtime, so it is written as `unknown`.

So the cell below computes a real md5 for each newly copied file and prints rows you can paste in as
an interim record. Treat them as provisional:

> **The next `python3 tools/drive_sync.py` run rewrites `MANIFEST.tsv` from Drive itself**, filling in
> the true `mime_type` and `drive_modified` and re-verifying every md5.
>
> **It does not touch `PENDING.tsv`.** No tool in this repo writes that file - it is maintained by
> hand. So `PENDING.tsv` is stale from the moment this notebook pushes, and stays stale, still naming
> the files this run just transferred, until someone edits it or regenerates it themselves.

The rows are printed and also saved to `/content/manifest-additions.tsv`, which is outside the clone
so it is never accidentally committed. It covers **this run's** copies only: cell 3 rewrites the state
file each time it runs, so if you copy in two passes, save the first fragment before re-running.


In [ ]:
# Cell 5 - md5 the new files and emit MANIFEST.tsv rows for them.
import hashlib
import json
import mimetypes
import os

STATE_PATH = "/content/drive_sync_colab_state.json"
REPO_DIR = "/content/repo"
DEST_ROOT = os.path.join(REPO_DIR, "drive")
FRAGMENT_PATH = "/content/manifest-additions.tsv"
CHUNK = 8 * 1024 * 1024

MANIFEST_COLUMNS = (
    "repo_path",
    "drive_id",
    "drive_title",
    "mime_type",
    "drive_size_bytes",
    "drive_modified",
    "md5",
    "status",
)

if not os.path.isfile(STATE_PATH):
    raise RuntimeError("No state file at {}. Run cell 3 (the copy cell) first.".format(STATE_PATH))
with open(STATE_PATH, encoding="utf-8") as handle:
    state = json.load(handle)

copied = state.get("copied", [])
if not copied:
    print("Nothing was copied, so there are no manifest rows to add.")
else:
    mimetypes.add_type("text/markdown", ".md")
    mimetypes.add_type("text/tab-separated-values", ".tsv")

    lines = []
    for item in sorted(copied, key=lambda entry: entry["repo_path"]):
        target = os.path.join(DEST_ROOT, item["repo_path"])
        digest = hashlib.md5()
        with open(target, "rb") as handle:
            for block in iter(lambda: handle.read(CHUNK), b""):
                digest.update(block)
        guessed = mimetypes.guess_type(item["repo_path"])[0] or "application/octet-stream"
        lines.append(
            "\t".join(
                (
                    item["repo_path"],
                    item["drive_id"],
                    item["drive_title"],
                    guessed,
                    str(os.path.getsize(target)),
                    "unknown",
                    digest.hexdigest(),
                    "ok (colab copy; mime_type guessed, drive_modified not available)",
                )
            )
        )

    with open(FRAGMENT_PATH, "w", encoding="utf-8") as handle:
        handle.write("\n".join(lines) + "\n")

    print("# columns: {}".format("\t".join(MANIFEST_COLUMNS)))
    print("# {} row(s); also saved to {} (outside the clone, so it is not committed)".format(
        len(lines), FRAGMENT_PATH
    ))
    print("# md5 is real. mime_type is guessed from the extension and drive_modified is unknown -")
    print("# the next  python3 tools/drive_sync.py  run rewrites MANIFEST.tsv from Drive and fixes both.")
    print()
    for line in lines:
        print(line)


## 6. Commit and push

Stages **only the individual files cell 3 recorded** - by path, from the state file - commits with a
message naming the file count, pushes, and then destroys the credential file and unregisters the
credential helper. That is deliberately narrower than `git add drive`: the appendix cell can leave a
large shard tree lying around, and nothing that cell 4 did not size-verify should ever be swept into a
commit here.

The credential file is destroyed on the "nothing to commit" path too, so a run with no work left to do
does not leave your token sitting in `/content/.git-credentials`. The one case where it is kept is a
**failed push**, so that a retry does not have to re-prompt for the token.

Two things it refuses to do: commit when cell 4 has not verified the sizes, and commit when the clone
is not on `claude/google-drive-github-sync-3833rb`. "Nothing to commit" is treated as an ordinary
outcome, not an error - it just means everything was already present.


In [ ]:
# Cell 6 - stage, commit, push, then destroy the credential file.
import json
import os
import subprocess

STATE_PATH = "/content/drive_sync_colab_state.json"
REPO_DIR = "/content/repo"
BRANCH = "claude/google-drive-github-sync-3833rb"
CRED_FILE = "/content/.git-credentials"

# Cell 2 defines scrub() over the token it read; this fallback keeps the cell runnable
# on its own in a fresh kernel, where there is no token in this process to leak anyway.
scrub = globals().get("scrub", lambda text: text or "")

if not os.path.isfile(STATE_PATH):
    raise RuntimeError("No state file at {}. Run cells 3 and 4 first.".format(STATE_PATH))
with open(STATE_PATH, encoding="utf-8") as handle:
    state = json.load(handle)

if not state.get("verified"):
    raise RuntimeError(
        "Cell 4 has not verified this copy (verified is false in {}). Run the verify cell and "
        "resolve every size mismatch before committing.".format(STATE_PATH)
    )


def git(*args, check=True):
    """Run git in the clone. Output is scrubbed, in case git echoes a URL back at us."""
    proc = subprocess.run(
        ("git", "-C", REPO_DIR) + args, text=True, capture_output=True
    )
    if check and proc.returncode != 0:
        raise RuntimeError(
            "git {} failed (exit {}):\n{}\n{}".format(
                scrub(" ".join(args)),
                proc.returncode,
                scrub(proc.stdout).strip(),
                scrub(proc.stderr).strip(),
            )
        )
    return proc


def destroy_credentials():
    """Unregister the credential helper and delete the token file."""
    subprocess.run(
        ("git", "config", "--global", "--unset", "credential.helper"), text=True, capture_output=True
    )
    if os.path.isfile(CRED_FILE):
        os.remove(CRED_FILE)
    print("Credential file {} removed and the global credential helper unset.".format(CRED_FILE))


branch_now = git("rev-parse", "--abbrev-ref", "HEAD").stdout.strip()
if branch_now != BRANCH:
    raise RuntimeError(
        "clone is on branch {!r}, expected {!r}. Run:  !git -C {} checkout {}".format(
            branch_now, BRANCH, REPO_DIR, BRANCH
        )
    )

# Stage the individual files this run recorded, by path - never the whole drive/ tree.
# `git add -- drive` would also sweep in anything else living under drive/, notably a
# several-hundred-MB shard tree from the appendix cell, which cell 4 never verified.
paths = sorted(
    {"drive/" + item["repo_path"] for item in state.get("copied", [])}
    | {"drive/" + repo_path for repo_path in state.get("present", [])}
)
if paths:
    git("add", "--", *paths)
staged = git("diff", "--cached", "--name-only").stdout.strip()
if not staged:
    print("Nothing to commit - every file in PENDING.tsv was already present in the clone.")
    print("If you expected changes, re-read cell 3's NOT FOUND lines.")
    # Nothing is left to retry on this path, so the token must not be left on disk.
    destroy_credentials()
else:
    count = len(staged.splitlines())
    copied_count = len(state.get("copied", []))
    message = (
        "drive: add {} file(s) mirrored from Google Drive\n\n"
        "Copied from a mounted Drive by tools/drive_sync_colab.ipynb ({} file(s) transferred in "
        "this run). Sizes verified against drive/PENDING.tsv. Neither inventory is updated here: "
        "the next tools/drive_sync.py run rewrites MANIFEST.tsv from Drive, but nothing generates "
        "PENDING.tsv - it is hand-maintained, and still lists these files until someone edits "
        "it.".format(
            count, copied_count
        )
    )
    git("commit", "-m", message)
    print("Committed {} file(s):".format(count))
    print(scrub(git("log", "-1", "--stat", "--pretty=%h %s").stdout))

    push = git("push", "origin", BRANCH, check=False)
    print(scrub(push.stdout).strip())
    print(scrub(push.stderr).strip())
    if push.returncode != 0:
        raise RuntimeError(
            "Push failed (exit {}). The commit is safe in {} - nothing is lost. "
            "See the last markdown cell: usually this is a non-fast-forward, fixed with "
            "git pull --rebase. A 403 instead means the token lacks repo/Contents:write "
            "or has expired.".format(push.returncode, REPO_DIR)
        )
    print("\nPushed to origin/{}.".format(BRANCH))

    # Kept only when the push FAILED (that raises above), so a retry does not re-prompt.
    destroy_credentials()


## 7. If the push was rejected, and cleaning up

### `! [rejected] ... (fetch first)` / `(non-fast-forward)`

Someone (or another route) pushed to the branch after cell 2 cloned it. Rebase your commit on top of
theirs and push again:

```python
!git -C /content/repo pull --rebase origin claude/google-drive-github-sync-3833rb
!git -C /content/repo push origin claude/google-drive-github-sync-3833rb
```

If the rebase reports a conflict, it will be in a mirrored file, which means the same Drive file was
transferred twice by different routes. Both copies should be byte-identical; if they are not, keep the
one whose md5 matches `drive/MANIFEST.tsv` and finish with `git rebase --continue`. To abandon the
attempt entirely: `git -C /content/repo rebase --abort`.

If the push was cleaned up by cell 6 before it failed, re-running cell 2 re-prompts for the token and
restores the credential helper; the commit in `/content/repo` is untouched.

### `remote: Permission ... denied` / `403`

The token lacks `repo` scope (classic) or *Contents: read and write* on this repository
(fine-grained), or it has expired. Issue a new one and re-run cells 2 and 6.

### Revoke the token

**Do this now if the token was short-lived or you created it just for this run.**
<https://github.com/settings/tokens> -> the token -> *Delete*. Cell 6 removes
`/content/.git-credentials` after a successful push, and the whole Colab VM (with any copy of the
token) is destroyed when the runtime is recycled - but a token that still exists on GitHub is still a
live credential. Revoking it is the only thing that actually ends its power.

Use *Runtime -> Disconnect and delete runtime* when you are done, and unmount with
`google.colab.drive.flush_and_unmount()` if you want the Drive session closed immediately.

### Then, at some point

Run `python3 tools/drive_sync.py` once from a machine that has the OAuth client
(`docs/DRIVE-SYNC.md`, Route 2). It rewrites `drive/MANIFEST.tsv` with true mime types, Drive
modification times and verified md5s.

`drive/PENDING.tsv` is **not** rewritten by that run, or by anything else - no tool in this repo
generates it. This notebook leaves it stale, still listing the files it just transferred, and someone
has to edit it by hand (or rebuild it from `MANIFEST.tsv` plus a Drive listing) to bring it back into
step.


---

## Appendix - the two 370 MiB `conversations.json` exports

`Claude Chats` holds two files, both titled `conversations.json`, both **exactly 388,264,753 bytes**.
Identical titles and identical byte counts make them near-certainly the same export saved twice.

They cannot enter this repo as ordinary git objects: GitHub hard-rejects any single file over 100 MB,
and the limit applies to the blob anywhere in history, so committing one and deleting it later does
not help. Cell 3 skips both rows for that reason.

The cell below commits nothing on its own. It has two modes:

* `"checksum"` (default) - md5 every copy the mount exposes and say whether they are genuinely the
  same bytes. Around 370 MiB read per copy over the Drive mount, so expect a few minutes.
* `"shard"` - hand the export to **`tools/shard_conversations.py`** in the clone, which splits it into
  one small JSON file per conversation under `YYYY-MM/` with an `INDEX.tsv`, rather than into opaque
  byte-range parts. Many small readable files are what a code-graph indexer can use; a reassemble-only
  blob is not. The mode always runs `--dry-run` first and only writes if you set
  `SHARD_CONFIRM = True`. **It writes to `/content/chats-shards`, outside the clone**, so several
  hundred MB of shards cannot be swept into a commit by cell 6; moving them into `drive/chats` is a
  separate, deliberate step the cell prints for you.

Even sharded, the conversation set adds its full weight to every future clone. `docs/REPO-SIZE.md`
argues for the cheaper answers first: a GitHub **release asset** (2 GB per asset, never in history),
Git LFS, or simply leaving the export in Drive and recording it in `MANIFEST.tsv` as external.


In [ ]:
# Appendix - checksum (or optionally shard) the two oversize conversations.json exports.
import hashlib
import os
import shlex

APPENDIX_MODE = "checksum"  # "checksum" or "shard"
SHARD_CONFIRM = False  # "shard" only dry-runs until this is True
# Deliberately OUTSIDE the clone. Cell 6 stages only the paths it recorded, but keeping
# several hundred MB of shards out of drive/ entirely means no future edit to cell 6 can
# commit them by accident either. Moving them in is a separate, deliberate step (printed below).
SHARD_OUT = "/content/chats-shards"
SHARD_SCRIPT = "/content/repo/tools/shard_conversations.py"
CHUNK = 8 * 1024 * 1024

DRIVE_ROOT = "/content/drive/MyDrive"
CHATS_DIR_COMPONENTS = ("The Method Materials", "Claude Chats")
TITLE = "conversations.json"

if APPENDIX_MODE not in ("checksum", "shard"):
    raise RuntimeError("APPENDIX_MODE must be 'checksum' or 'shard', not {!r}".format(APPENDIX_MODE))
if not os.path.isdir(DRIVE_ROOT):
    raise RuntimeError("Drive is not mounted at {}. Run cell 1 first.".format(DRIVE_ROOT))


def norm_name(name):
    return name.strip().strip(".").casefold()


folder = DRIVE_ROOT
for component in CHATS_DIR_COMPONENTS:
    entries = [
        entry for entry in os.listdir(folder) if norm_name(entry) == norm_name(component)
    ]
    if len(entries) != 1:
        raise RuntimeError(
            "expected exactly one {!r} inside {!r}, found {}: {}".format(
                component, folder, len(entries), sorted(entries)
            )
        )
    folder = os.path.join(folder, entries[0])

copies = sorted(
    os.path.join(folder, entry)
    for entry in os.listdir(folder)
    if norm_name(entry) == norm_name(TITLE) and os.path.isfile(os.path.join(folder, entry))
)
if not copies:
    raise RuntimeError("no file named {!r} in {!r}".format(TITLE, folder))

print("{} copy/copies of {!r} on the mount:".format(len(copies), TITLE))
for path in copies:
    print("  {}  ({:,} bytes)".format(path, os.path.getsize(path)))
if len(copies) == 1:
    print(
        "\nOnly one copy is visible. PENDING.tsv lists two Drive files with this title; a mounted\n"
        "filesystem cannot show two entries with the same name in one folder, so the second is\n"
        "reachable only through the Drive API (tools/drive_sync.py) or the Drive web UI."
    )


def md5_of(path):
    digest = hashlib.md5()
    read = 0
    size = os.path.getsize(path)
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(CHUNK), b""):
            digest.update(block)
            read += len(block)
            print("\r  {:.0f}% of {}".format(100.0 * read / size, os.path.basename(path)), end="")
    print()
    return digest.hexdigest()


if APPENDIX_MODE == "checksum":
    print("\nHashing (this reads every byte over the Drive mount - minutes, not seconds):")
    digests = [(path, md5_of(path)) for path in copies]
    for path, digest in digests:
        print("  {}  {}".format(digest, path))
    if len(digests) > 1:
        unique = {digest for _, digest in digests}
        if len(unique) == 1:
            print("\nIDENTICAL - the copies are the same bytes. Delete one in Drive, keep the other.")
        else:
            print("\nDIFFERENT - same size, different content. Both are real and must be kept apart.")
    print("\nNothing was written and nothing was committed.")
else:
    import subprocess
    import sys

    if not os.path.isfile(SHARD_SCRIPT):
        raise RuntimeError(
            "{} is not in the clone. Run cell 2 first; if it is still missing, this branch does "
            "not carry the sharding script yet and there is nothing to delegate to.".format(
                SHARD_SCRIPT
            )
        )

    source = copies[0]
    # ijson lets the script stream the export at flat memory; without it json.load needs
    # several GB for a 370 MiB file. The script warns loudly if it has to fall back.
    subprocess.run(
        (sys.executable, "-m", "pip", "install", "--quiet", "ijson"), check=False
    )

    command = [sys.executable, SHARD_SCRIPT, source, "--out", SHARD_OUT]
    dry = command + ["--dry-run"]
    print("\n$ {}".format(shlex.join(dry)))  # quoted, so it is copy-pasteable
    proc = subprocess.run(dry, text=True)
    if proc.returncode != 0:
        raise RuntimeError(
            "shard_conversations.py --dry-run exited {}. Read its output above; nothing was "
            "written.".format(proc.returncode)
        )

    if not SHARD_CONFIRM:
        print(
            "\nDry run only. Nothing was written.\n"
            "Set SHARD_CONFIRM = True at the top of this cell to write into {}, then re-run.\n"
            "Read docs/REPO-SIZE.md first - these files are permanent in git history.".format(
                SHARD_OUT
            )
        )
    else:
        print("\n$ {}".format(shlex.join(command)))
        proc = subprocess.run(command, text=True)
        if proc.returncode != 0:
            raise RuntimeError(
                "shard_conversations.py exited {}. Read its output above.".format(proc.returncode)
            )
        print(
            "\nWritten to {}, which is OUTSIDE the clone - nothing here is staged or committed,\n"
            "and cell 6 stages only the files it copied from PENDING.tsv, so it cannot pick these\n"
            "up either. Read docs/REPO-SIZE.md, then move and commit them deliberately:\n"
            "  !cp -a {} /content/repo/drive/chats\n"
            "  !{}\n"
            "  !{}\n"
            "The credential helper is unset after cell 6 succeeds, so re-run cell 2 before "
            "pushing.".format(
                SHARD_OUT,
                shlex.quote(SHARD_OUT),
                shlex.join(["git", "-C", "/content/repo", "add", "--", "drive/chats"]),
                shlex.join(
                    ["git", "-C", "/content/repo", "commit", "-m",
                     "drive: shard conversations export"]
                ),
            )
        )
